In [ ]:
from pathlib import Path
import pandas as pd

EXPERIMENT_NAME = "5c6bf125-6e0f-41be-a447-b03ff294c9cc"
BASE_PATH = Path("/Users/pmihack/claire/hs-array/data/D28_AB_Rep1")
# nuclei_features now live under the run-ID system (run_..._cyq9 = WARD00034 R2, post MIN_AREA fix)
PARQUET_PATH = Path("/Users/pmihack/claire/hs-array/outputs/run_20260806_141727_cyq9/nuclei_features/5c6bf125-6e0f-41be-a447-b03ff294c9cc_nuclei_features.parquet")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define pseudocolor mapping rules - helpful to generate images on the fly
pseudocolor_map = {
    "DAPI": "blue",
    "Brightfield": "gray",
    "Alexa 488": "green",
    "CF568": "red",
    "Alexa 647": "magenta"
}

# Define pseudocolor mapping rules for matplotlib - helpful to generate images on the fly
mpl_colormaps = {
    "blue": LinearSegmentedColormap.from_list("black_blue", [(0, 0, 0), (0, 0, 1)]),
    "green": LinearSegmentedColormap.from_list("black_green", [(0, 0, 0), (0, 1, 0)]),
    "red": LinearSegmentedColormap.from_list("black_red", [(0, 0, 0), (1, 0, 0)]),
    "magenta": LinearSegmentedColormap.from_list("black_magenta", [(0, 0, 0), (1, 0, 1)]),
    "gray": LinearSegmentedColormap.from_list("black_gray", [(0, 0, 0), (1, 1, 1)])
}

In [ ]:
import re
import string
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

# Namespace consistent across all experiments
NS = {'h': '43B2A954-E3C3-47E1-B392-6635266B0DD3/HarmonyV7'}
# ponytail: BASE_PATH is set in cell 1; removed local override here

# XML helpers
def _find_text(el, tag):
    """Return text of a matched subelement, or None if not found."""
    match = el.find(tag, NS)
    return match.text if match is not None else None


def parse_experiment_xml(experiment_path: Path) -> dict:
    """Extract measurement ID, date, and plate name from the experiment XML."""
    xml_file = next(experiment_path.glob("*.xml"), None)
    if xml_file is None:
        raise FileNotFoundError(f"No experiment XML found in {experiment_path}")

    root = ET.parse(xml_file).getroot()
    return {
        "measurement_id": _find_text(root, 'h:MeasurementID'),
        "date":           _find_text(root, 'h:Date'),
        "plate":          _find_text(root, 'h:InitialPlateName'),
    }


def parse_index_xml(experiment_path: Path) -> dict:
    """Extract plate ID, resolution, and channel info from the index XML."""
    index_xml = next((experiment_path / "index").glob("*.xml"), None)
    if index_xml is None:
        raise FileNotFoundError(f"No index XML found in {experiment_path / 'index'}")

    root = ET.parse(index_xml).getroot()

    plate_id = _find_text(root, './/h:PlateID')
    x_res = float(_find_text(root, './/h:ImageResolutionX')) * 1e6
    y_res = float(_find_text(root, './/h:ImageResolutionY')) * 1e6
    channel_df = _parse_channels(root)

    return {"plate_id": plate_id, "x_res": x_res, "y_res": y_res, "channel_df": channel_df}


def _parse_channels(root) -> pd.DataFrame:
    """Extract channel metadata from the index XML root into a DataFrame."""
    channels = []
    for map_el in root.findall(".//h:Map", NS):
        first_entry = map_el.find("h:Entry", NS)
        if first_entry is None or first_entry.find("h:ChannelName", NS) is None:
            continue
        for entry in map_el.findall("h:Entry", NS):
            ch_id = entry.attrib.get("ChannelID")
            channels.append({
                "ChannelID":     int(ch_id) if ch_id is not None else None,
                "Channel_name":  _find_text(entry, "h:ChannelName"),
                "Type":          _find_text(entry, "h:ChannelType"),
                "Excitation_nm": _find_text(entry, "h:MainExcitationWavelength"),
                "Emission_nm":   _find_text(entry, "h:MainEmissionWavelength"),
            })
        break  # Only process the first matching Map

    return pd.DataFrame(channels).sort_values("ChannelID").reset_index(drop=True)


# Image file parsing
def _parse_filename(name: str) -> list:
    """Parse r/c/f/p/ch/t components from an Opera Phenix TIFF filename."""
    match = re.match(r"r(\d+)c(\d+)f(\d+)p(\d+)-ch(\d+)t(\d+)", name)
    return [int(g) for g in match.groups()] if match else [None] * 6


def build_image_df(img_dir: Path) -> pd.DataFrame:
    """Collect all TIFFs under img_dir and parse their filename metadata."""
    files = sorted(f for f in img_dir.rglob("*") if f.suffix.lower() == ".tiff")
    df = pd.DataFrame({
        "filepath":     files,
        "filename":     [f.name for f in files],
        "subdirectory": [f.parent.relative_to(img_dir) for f in files],
    })
    df[["Row", "Column", "Frame", "Plane", "ChannelID", "Time"]] = (
        df["filename"].apply(lambda x: pd.Series(_parse_filename(x)))
    )
    return df


# Top-level loader 
def load_experiment(experiment_name: str, pseudocolor_map: dict, mpl_colormaps: dict) -> dict:
    """
    Load all metadata and image file info for a given experiment.

    Parameters
    ----------
    experiment_name : str
        Experiment UUID / folder name under BASE_PATH.
    pseudocolor_map : dict
        Maps channel names to pseudocolor strings (e.g. {"DAPI": "blue"}).
    mpl_colormaps : dict
        Maps pseudocolor strings to matplotlib colormap names.

    Returns
    -------
    dict with keys: experiment_meta, index_meta, image_df, merged_df, summary
    """
    experiment_path = BASE_PATH / experiment_name

    experiment_meta = parse_experiment_xml(experiment_path)
    index_meta = parse_index_xml(experiment_path)

    channel_df = index_meta["channel_df"].copy()
    channel_df["Pseudocolor"]    = channel_df["Channel_name"].map(pseudocolor_map).fillna("gray")
    channel_df["MPL_colormap"]   = channel_df["Pseudocolor"].str.lower().map(mpl_colormaps)
    channel_df["Measurement_ID"] = experiment_meta["measurement_id"]
    channel_df["Measurement_date"] = experiment_meta["date"]
    channel_df["Plate_ID"]       = index_meta["plate_id"]
    channel_df["res_x"]          = index_meta["x_res"]
    channel_df["res_y"]          = index_meta["y_res"]

    img_dir = experiment_path / "images"
    image_df = build_image_df(img_dir)
    merged_df = pd.merge(image_df, channel_df, on="ChannelID")

    summary = {
        "measurement_id": experiment_meta["measurement_id"],
        "plate":          experiment_meta["plate"],
        "wells":          merged_df[["Row", "Column"]].drop_duplicates().shape[0],
        "channels":       merged_df["ChannelID"].nunique(),
        "z_planes":       merged_df["Plane"].nunique(),
        "frames":         merged_df["Frame"].nunique(),
        "timepoints":     merged_df["Time"].nunique(),
    }

    print(f"""
        Experiment ID:       {summary['measurement_id']}
        Plate ID:            {summary['plate']}
        Wells imaged:        {summary['wells']}
        Frames per well:     {summary['frames']}
        Channels per image:  {summary['channels']}
        Z-slices per image:  {summary['z_planes']}
        Timepoints:          {summary['timepoints']}
        """)

    return {
        "experiment_meta": experiment_meta,
        "index_meta":      index_meta,
        "channel_df":      channel_df,
        "image_df":        image_df,
        "merged_df":       merged_df,
        "summary":         summary,
    }

## Images that failed nucleus segmentation
Some images failed nucleus segmentation, we wanted to verify that there were no nuclei within these frames\
Experiment 'c8f2dbbe-6632-4063-ab68-0b4fa37909af' has the most DAPI frames that do not get segmented, so is a good example set to work with

In [ ]:
experiment_name = EXPERIMENT_NAME

result = load_experiment(experiment_name, pseudocolor_map, mpl_colormaps)

channel_df = result["channel_df"]
merged_df  = result["merged_df"]
summary    = result["summary"]

In [ ]:
import matplotlib.pyplot as plt
import tifffile
import numpy as np

# Load nucleus features parquet (Row, Column, Frame identify each segmented site)
nuc_features = pd.read_parquet(PARQUET_PATH)

# DAPI only (ChannelID == 1)
dapi_df = merged_df[merged_df["ChannelID"] == 1].copy()

# Sites present in the image manifest vs. the parquet
manifest_sites = set(zip(dapi_df["Row"], dapi_df["Column"], dapi_df["Frame"]))
parquet_sites  = set(zip(nuc_features["Row"], nuc_features["Column"], nuc_features["Frame"]))

missing_sites = sorted(manifest_sites - parquet_sites)
print(f"Sites with no segmented nuclei: {len(missing_sites)} / {len(manifest_sites)}")

# Show up to 5 missing sites as DAPI MIPs
for (row, col, frame) in missing_sites[:5]:
    planes = (
        dapi_df[(dapi_df["Row"] == row) & (dapi_df["Column"] == col) & (dapi_df["Frame"] == frame)]
        .sort_values("Plane")["filepath"]
        .tolist()
    )
    mip = np.max(np.stack([tifffile.imread(p) for p in planes]), axis=0)

    plt.figure(figsize=(8, 8))
    plt.imshow(mip, cmap="gray_r")
    plt.title(f"r{row:02d}c{col:02d}f{frame:02d} — DAPI MIP (no nuclei segmented)")
    plt.axis("off")
    plt.show()

In [6]:
merged_df

,filepath,filename,subdirectory,Row,Column,Frame,Plane,ChannelID,Time,Channel_name,Type,Excitation_nm,Emission_nm,Pseudocolor,MPL_colormap,Measurement_ID,Measurement_date,Plate_ID,res_x,res_y
0,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r01c01f01p01-ch01t01.tiff,r01c01,1,1,1,1,1,1,DAPI,Fluorescence,375,456,blue,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
1,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r01c01f01p01-ch02t01.tiff,r01c01,1,1,1,1,2,1,Alexa 647,Fluorescence,640,706,magenta,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
2,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r01c01f01p01-ch03t01.tiff,r01c01,1,1,1,1,3,1,Alexa 488,Fluorescence,488,522,green,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
3,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r01c01f01p01-ch04t01.tiff,r01c01,1,1,1,1,4,1,Alexa 568,Fluorescence,561,599,red,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
4,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r01c01f02p01-ch01t01.tiff,r01c01,1,1,2,1,1,1,DAPI,Fluorescence,375,456,blue,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53755,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r16c24f34p01-ch04t01.tiff,r16c24,16,24,34,1,4,1,Alexa 568,Fluorescence,561,599,red,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
53756,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r16c24f35p01-ch01t01.tiff,r16c24,16,24,35,1,1,1,DAPI,Fluorescence,375,456,blue,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
53757,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r16c24f35p01-ch02t01.tiff,r16c24,16,24,35,1,2,1,Alexa 647,Fluorescence,640,706,magenta,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187
53758,/data/CARDPB2/iNDI/Production/c170959e-633a-45...,r16c24f35p01-ch03t01.tiff,r16c24,16,24,35,1,3,1,Alexa 488,Fluorescence,488,522,green,<matplotlib.colors.LinearSegmentedColormap obj...,c170959e-633a-458c-8542-2aa619c0d7f8,2026-02-13T10:15:19.8992903-05:00,INDI00035D,0.094187,0.094187


In [ ]:
from scipy.ndimage import gaussian_laplace
from skimage import filters, morphology
from scipy import ndimage as ndi
import numpy as np
from scipy.stats import norm

subset = merged_df[
    (merged_df['Row'] == 3) &
    (merged_df['Column'] == 3) &
    (merged_df['Frame'] == 1)]

print(subset)

# Collect all 10 DAPI planes and max-project
dapi_planes = (
    subset[subset["Channel_name"] == "DAPI"]
    .sort_values("Plane")["filepath"]
    .tolist()
)
nuc = np.max(np.stack([tifffile.imread(p) for p in dapi_planes]), axis=0)
print(f"MIP shape: {nuc.shape}, dtype: {nuc.dtype}")

# Parameters - same as AICS standard
intensity_scaling_param = [1, 7]
blur_sigma = 1
min_area = 3500

# Normalize to normal distribution
m, s = norm.fit(nuc.flatten())
stretch_min = max(m - intensity_scaling_param[0] * s, nuc.min())
stretch_max = min(m + intensity_scaling_param[1] * s, nuc.max())
nuc_stretch = np.clip(nuc, stretch_min, stretch_max)
image_norm = (nuc_stretch - stretch_min) / (stretch_max - stretch_min)

# Blur image
blurred = filters.gaussian(image_norm, sigma=blur_sigma)

# step 1: low level thresholding
triangle_cutoff = filters.threshold_triangle(blurred)
global_median_cutoff = np.percentile(blurred, 50)

th_low_cutoff = (triangle_cutoff + global_median_cutoff) / 2
img_low_level = blurred > th_low_cutoff

img_low_level = morphology.remove_small_objects(img_low_level, min_size=min_area, connectivity=1, out=img_low_level)
img_low_level = morphology.dilation(img_low_level, footprint=morphology.disk(2))

# step 2: high level thresholding
otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
img_high_level = np.zeros_like(img_low_level)
lab_low, num_obj = morphology.label(img_low_level, return_num=True, connectivity=1)
for idx in range(num_obj):
    single_obj = lab_low == (idx + 1)
    local_otsu = filters.threshold_otsu(blurred[single_obj])
    if local_otsu > otsu_cutoff:
        img_high_level[np.logical_and(blurred > 0.98 * local_otsu, single_obj)] = 1

filled = ndi.binary_fill_holes(img_high_level)
filled = morphology.dilation(filled, footprint=morphology.disk(2))
labeled_filled = morphology.label(filled, connectivity=1)
nuc_seg = morphology.remove_small_objects(labeled_filled, min_size=min_area)

fig, axs = plt.subplots(1, 6, figsize=(20, 5))
fig.suptitle("WARD00034-R2 r03c03f01 (MIP)")

axs[0].imshow(nuc, cmap='gray')
axs[0].set_title("DAPI - NUCLEUS")
axs[0].axis("off")

axs[1].imshow(image_norm, cmap='gray')
axs[1].set_title("NORMALIZED IMAGE")
axs[1].axis("off")

axs[2].imshow(blurred, cmap='gray')
axs[2].set_title("BLURRED")
axs[2].axis("off")

axs[3].imshow(img_low_level)
axs[3].set_title("LOW LEVEL THRESHOLD")
axs[3].axis("off")

axs[4].imshow(img_high_level)
axs[4].set_title("HIGH LEVEL THRESHOLD")
axs[4].axis("off")

axs[5].imshow(nuc_seg)
axs[5].set_title("FINAL CLEANED")
axs[5].axis("off")

plt.tight_layout()
plt.show()

fig, axs = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("WARD00034-R2 r03c03f01 (MIP)")

axs[0].imshow(nuc, cmap='gray')
axs[0].set_title("DAPI - NUCLEUS")
axs[0].axis("off")

axs[1].imshow(image_norm, cmap='gray')
axs[1].set_title("NORMALIZED IMAGE")
axs[1].axis("off")

axs[2].imshow(nuc_seg)
axs[2].set_title("FINAL CLEANED")
axs[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
## QC grid — 12 random sites (DAPI MIP + segmentation overlay)

def _segment_mip(mip, min_area=3500):
    """Same AICS-style segmentation used in the walkthrough cell above."""
    m, s = norm.fit(mip.flatten())
    stretch_min = max(m - s, mip.min())
    stretch_max = min(m + 7 * s, mip.max())
    norm_img = (np.clip(mip, stretch_min, stretch_max) - stretch_min) / (stretch_max - stretch_min)
    blurred = filters.gaussian(norm_img, sigma=1)

    th_low = (filters.threshold_triangle(blurred) + np.percentile(blurred, 50)) / 2
    low = morphology.remove_small_objects(blurred > th_low, min_size=min_area, connectivity=1)
    low = morphology.dilation(low, footprint=morphology.disk(2))

    otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
    lab_low, num_obj = morphology.label(low, return_num=True, connectivity=1)
    high = np.zeros_like(low)
    for idx in range(num_obj):
        obj = lab_low == (idx + 1)
        local_otsu = filters.threshold_otsu(blurred[obj])
        if local_otsu > otsu_cutoff:
            high[np.logical_and(blurred > 0.98 * local_otsu, obj)] = 1

    filled = ndi.binary_fill_holes(high)
    filled = morphology.dilation(filled, footprint=morphology.disk(2))
    seg = morphology.remove_small_objects(morphology.label(filled, connectivity=1), min_size=min_area)
    return seg


_dapi = merged_df[merged_df["ChannelID"] == 1].copy()

# Sample 12 random sites from the parquet (sites that have nuclei)
_sites = nuc_features[["Row", "Column", "Frame"]].drop_duplicates()
_sample = _sites.sample(n=min(12, len(_sites)), random_state=42).reset_index(drop=True)

fig, axs = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle("WARD00034-R2 — 12 random sites (DAPI MIP + segmentation overlay)", fontsize=14)

for i, site in _sample.iterrows():
    r, c, f = int(site["Row"]), int(site["Column"]), int(site["Frame"])
    ax = axs[i // 4, i % 4]

    planes = (
        _dapi[(_dapi["Row"] == r) & (_dapi["Column"] == c) & (_dapi["Frame"] == f)]
        .sort_values("Plane")["filepath"]
        .tolist()
    )
    if not planes:
        ax.set_visible(False)
        continue

    mip = np.max(np.stack([tifffile.imread(p) for p in planes]), axis=0)
    seg = _segment_mip(mip)

    ax.imshow(mip, cmap="gray_r")
    if seg.max() > 0:
        overlay = np.zeros((*mip.shape, 4), dtype=float)
        overlay[seg > 0] = [0, 1, 0, 0.3]  # green tint
        ax.imshow(overlay)
    ax.set_title(f"r{r:02d}c{c:02d}f{f:02d}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()